In [0]:
%sql
CREATE OR REPLACE TABLE main.gh_archive.Fact_Push_Activity AS
SELECT 
    event_id,
    repo_id,
    actor_id,
    branch_name,
    created_at as push_timestamp
FROM main.gh_archive.silver_events
WHERE type = 'PushEvent';

In [0]:
%sql
CREATE OR REPLACE TABLE main.gh_archive.Dim_Repo AS
WITH ranked_repos AS (
    SELECT 
        repo_id,
        repo_name,
        repo_url,
        ROW_NUMBER() OVER (PARTITION BY repo_id ORDER BY created_at DESC) AS rn
    FROM main.gh_archive.silver_events
)
SELECT 
    repo_id,
    repo_name,
    repo_url
FROM ranked_repos
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE main.gh_archive.Dim_Actor AS
WITH ranked_actors AS (
    SELECT 
        actor_id,
        display_login,
        ROW_NUMBER() OVER (PARTITION BY actor_id ORDER BY created_at DESC) AS rn
    FROM main.gh_archive.silver_events
)
SELECT 
    DISTINCT actor_id,
    display_login
FROM ranked_actors
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE main.gh_archive.Fact_PR_Lifecycle AS
WITH pr_lifecycle AS (
    SELECT
        repo_id,
        pr_number,
        MAX(CASE WHEN action = 'opened' THEN actor_id END) AS actor_id,
        MAX(CASE WHEN action = 'opened' THEN created_at END) AS opened_at,
        MAX(CASE WHEN action IN ('closed', 'merged') THEN created_at END) AS closed_at,
        MAX(CASE WHEN action IN ('closed', 'merged') THEN action END) AS state
    FROM main.gh_archive.silver_events
    WHERE type = 'PullRequestEvent'
    GROUP BY repo_id, pr_number
)
SELECT
    repo_id,
    pr_number,
    actor_id,
    CAST(opened_at AS TIMESTAMP) AS opened_at,
    CAST(closed_at AS TIMESTAMP) AS closed_at,
    state,
    TIMESTAMPDIFF(HOUR, CAST(opened_at AS TIMESTAMP), CAST(closed_at AS TIMESTAMP)) AS cycle_time_hours
FROM pr_lifecycle
WHERE closed_at IS NOT NULL